# Modelagem e Avaliação (v2)
## Comparação: Features v1 vs Features v2 (ELO + Amistosos/Competitivos)

**Objetivo:** Avaliar se as novas features (ELO ponderado e separação amistoso/competitivo) melhoram o desempenho dos modelos em relação à versão anterior.

**Datasets:**
- `data/processed/features_completo.csv` — versão original (7 features)
- `data/processed/features_completo_v2.csv` — versão melhorada (13 features)

**Divisão temporal:**
- Treino: Copas 1994–2018 (216 amostras)
- Teste: Copa 2022 (32 amostras)

## 1. Imports e Carregamento

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from xgboost import XGBRegressor
from scipy.stats import wilcoxon

sns.set_theme(style='whitegrid')
np.random.seed(42)

df_v1 = pd.read_csv('../data/processed/features_completo.csv')
df_v2 = pd.read_csv('../data/processed/features_completo_v2.csv')

print(f'Dataset v1: {df_v1.shape}')
print(f'Dataset v2: {df_v2.shape}')

## 2. Definição das Features e Divisão Treino/Teste

In [ ]:
features_v1 = [
    'media_gols_marcados_ciclo', 'media_gols_sofridos_ciclo',
    'pct_vitorias_ciclo', 'total_jogos_ciclo',
    'media_gols_marcados_ult15', 'media_gols_sofridos_ult15',
    'pct_vitorias_ult15'
]

features_v2 = features_v1 + [
    'media_gols_competitivos', 'media_gols_sofridos_comp',
    'pct_vitorias_comp', 'media_gols_amistosos',
    'gols_ponderados_elo_ciclo', 'gols_ponderados_elo_ult15'
]

def dividir(df, features):
    X_train = df[df['copa_alvo'] < 2022][features]
    X_test  = df[df['copa_alvo'] == 2022][features]
    y_train = df[df['copa_alvo'] < 2022]['media_gols_copa']
    y_test  = df[df['copa_alvo'] == 2022]['media_gols_copa']
    return X_train, X_test, y_train, y_test

X_train_v1, X_test_v1, y_train, y_test = dividir(df_v1, features_v1)
X_train_v2, X_test_v2, _, _            = dividir(df_v2, features_v2)

print(f'Treino: {X_train_v1.shape[0]} amostras | Teste: {X_test_v1.shape[0]} amostras')
print(f'Features v1: {len(features_v1)} | Features v2: {len(features_v2)}')

## 3. Treino e Avaliação — V1 vs V2

Treinamos os 3 modelos com cada versão das features e comparamos o MAE e RMSE no teste.

In [ ]:
def treinar_avaliar(X_train, X_test, y_train, y_test, versao):
    resultados = []
    modelos_treinados = {}

    configs = [
        ('Regressão Linear', LinearRegression()),
        ('Random Forest',    RandomForestRegressor(n_estimators=100, random_state=42)),
        ('XGBoost',          XGBRegressor(n_estimators=100, random_state=42)),
    ]

    for nome, modelo in configs:
        modelo.fit(X_train, y_train)
        y_pred = modelo.predict(X_test)
        resultados.append({
            'Versão':  versao,
            'Modelo':  nome,
            'MAE':     mean_absolute_error(y_test, y_pred),
            'RMSE':    root_mean_squared_error(y_test, y_pred),
            'y_pred':  y_pred
        })
        modelos_treinados[nome] = modelo

    return resultados, modelos_treinados


res_v1, modelos_v1 = treinar_avaliar(X_train_v1, X_test_v1, y_train, y_test, 'v1')
res_v2, modelos_v2 = treinar_avaliar(X_train_v2, X_test_v2, y_train, y_test, 'v2')

df_resultados = pd.DataFrame(res_v1 + res_v2).drop(columns='y_pred')
df_resultados = df_resultados.sort_values(['Modelo', 'Versão']).reset_index(drop=True)

print('Comparação v1 vs v2 — Conjunto de Teste (Copa 2022):')
print(df_resultados.to_string(index=False))

## 4. Visualização — Comparação V1 vs V2

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, metrica in zip(axes, ['MAE', 'RMSE']):
    pivot = df_resultados.pivot(index='Modelo', columns='Versão', values=metrica)
    pivot.plot(kind='bar', ax=ax, color=['steelblue', 'coral'],
               edgecolor='black', alpha=0.85)
    ax.set_title(f'{metrica} — v1 vs v2')
    ax.set_ylabel(metrica)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=15)
    ax.legend(title='Versão')

plt.suptitle('Impacto das Novas Features (ELO + Amistosos/Competitivos)', fontsize=13)
plt.tight_layout()
plt.savefig('../article/figures/comparacao_v1_v2.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Validação Cruzada Temporal — V2

In [ ]:
copas_ordenadas = sorted(df_v2['copa_alvo'].unique())
folds = [(copas_ordenadas[:i], copas_ordenadas[i]) for i in range(2, len(copas_ordenadas))]

mae_folds_v2 = {'Regressão Linear': [], 'Random Forest': [], 'XGBoost': []}

for copas_treino, copa_teste in folds:
    mask_tr = df_v2['copa_alvo'].isin(copas_treino)
    mask_te = df_v2['copa_alvo'] == copa_teste

    Xtr = df_v2[mask_tr][features_v2]
    ytr = df_v2[mask_tr]['media_gols_copa']
    Xte = df_v2[mask_te][features_v2]
    yte = df_v2[mask_te]['media_gols_copa']

    for nome, modelo_cls in [
        ('Regressão Linear', LinearRegression()),
        ('Random Forest',    RandomForestRegressor(n_estimators=100, random_state=42)),
        ('XGBoost',          XGBRegressor(n_estimators=100, random_state=42))
    ]:
        modelo_cls.fit(Xtr, ytr)
        mae_folds_v2[nome].append(mean_absolute_error(yte, modelo_cls.predict(Xte)))

print('Validação Cruzada Temporal — Features v2:')
print(f'{"Modelo":<20} {"MAE Médio":>10} {"Desvio-Padrão":>15}')
print('-' * 47)
for modelo, maes in mae_folds_v2.items():
    print(f'{modelo:<20} {np.mean(maes):>10.4f} {np.std(maes):>15.4f}')

## 6. Importância de Features — V2

In [ ]:
rf_v2  = modelos_v2['Random Forest']
xgb_v2 = modelos_v2['XGBoost']

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

for ax, (modelo, nome) in zip(axes, [(rf_v2, 'Random Forest v2'), (xgb_v2, 'XGBoost v2')]):
    imp = pd.Series(modelo.feature_importances_, index=features_v2).sort_values()
    colors = ['coral' if 'elo' in f or 'compet' in f or 'amist' in f
              else 'steelblue' for f in imp.index]
    imp.plot(kind='barh', ax=ax, color=colors, edgecolor='black')
    ax.set_title(f'Importância de Features — {nome}')
    ax.set_xlabel('Importância')

# Legenda manual
from matplotlib.patches import Patch
legend = [Patch(color='coral', label='Features novas (v2)'),
          Patch(color='steelblue', label='Features originais (v1)')]
axes[0].legend(handles=legend, loc='lower right')

plt.tight_layout()
plt.savefig('../article/figures/importancia_features_v2.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Teste Estatístico — Wilcoxon

In [ ]:
# Comparar v1 vs v2 para o melhor modelo (Random Forest)
mae_v1_folds = []
mae_v2_folds = []

for copas_treino, copa_teste in folds:
    mask_tr_v1 = df_v1['copa_alvo'].isin(copas_treino)
    mask_te_v1 = df_v1['copa_alvo'] == copa_teste
    mask_tr_v2 = df_v2['copa_alvo'].isin(copas_treino)
    mask_te_v2 = df_v2['copa_alvo'] == copa_teste

    m1 = RandomForestRegressor(n_estimators=100, random_state=42)
    m1.fit(df_v1[mask_tr_v1][features_v1], df_v1[mask_tr_v1]['media_gols_copa'])
    mae_v1_folds.append(mean_absolute_error(
        df_v1[mask_te_v1]['media_gols_copa'],
        m1.predict(df_v1[mask_te_v1][features_v1])
    ))

    m2 = RandomForestRegressor(n_estimators=100, random_state=42)
    m2.fit(df_v2[mask_tr_v2][features_v2], df_v2[mask_tr_v2]['media_gols_copa'])
    mae_v2_folds.append(mean_absolute_error(
        df_v2[mask_te_v2]['media_gols_copa'],
        m2.predict(df_v2[mask_te_v2][features_v2])
    ))

try:
    stat, p = wilcoxon(mae_v1_folds, mae_v2_folds)
    print(f'Wilcoxon — Random Forest v1 vs v2:')
    print(f'  MAE médio v1: {np.mean(mae_v1_folds):.4f}')
    print(f'  MAE médio v2: {np.mean(mae_v2_folds):.4f}')
    print(f'  p-value: {p:.4f}')
    print(f'  Diferença significativa: {"Sim" if p < 0.05 else "Não"}')
except Exception as e:
    print(f'Erro no teste: {e}')

## 8. Tabela Comparativa Final

In [ ]:
resumo_final = pd.DataFrame({
    'Modelo':          ['Regressão Linear', 'Random Forest', 'XGBoost'] * 2,
    'Versão':          ['v1'] * 3 + ['v2'] * 3,
    'MAE Teste':       [r['MAE'] for r in res_v1] + [r['MAE'] for r in res_v2],
    'RMSE Teste':      [r['RMSE'] for r in res_v1] + [r['RMSE'] for r in res_v2],
    'MAE CV (média)':  [
        np.mean(mae_v1_folds), np.mean(mae_v1_folds), np.mean(mae_v1_folds),
        np.mean(mae_folds_v2['Regressão Linear']),
        np.mean(mae_folds_v2['Random Forest']),
        np.mean(mae_folds_v2['XGBoost'])
    ]
}).round(4)

print('Tabela Comparativa Final — v1 vs v2:')
print(resumo_final.to_string(index=False))

resumo_final.to_csv('../article/tables/comparacao_v1_v2.csv', index=False)
print('\nTabela salva em article/tables/comparacao_v1_v2.csv')

## 9. Conclusões

**Análise do impacto das novas features:**
- Comparar MAE v1 vs v2 para cada modelo e verificar se houve melhoria
- O teste de Wilcoxon confirma se a diferença é estatisticamente significativa
- A importância de features indica quais das novas variáveis (ELO, competitivos) mais contribuíram

**Modelo escolhido para previsão 2026:**
- Usar a versão (v1 ou v2) com melhor MAE na validação cruzada
- Em caso de empate estatístico, preferir o modelo mais simples (v1 — princípio da parcimônia)

**Próximo passo:** `04_previsao_2026.ipynb` — gerar previsões com o modelo final.